# 18 · Work IQ

## Goal

Ground "which of my projects touch this supplier" in the caller's real
business context — location, projects, recent Teams/email — via Work IQ
(GA June 2026), with the consent boundary and PII discipline as the actual
lesson, not the retrieval mechanics.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.checkpoint import checkpoint
checkpoint(
    name="Work IQ enabled for this environment/tenant",
    probe=lambda: input("Confirmed Work IQ is enabled (tenant admin setting)? (y/n): ") == "y",
    remediation="Enable Work IQ in the Power Platform admin center for this environment.",
)


## Concept

Work IQ (GA June 2026) surfaces a user's business context — location,
projects, recent Teams/email — via REST, A2A, or MCP. It's tempting to
treat this as "the agent now knows everything about the user"; it doesn't,
and shouldn't. The consent boundary is the actual lesson: Work IQ surfaces
what the *asking user's own* context contains, under their own consent —
never a colleague's, never a broader search. `workiq-02-consent-boundary`
in the golden set is the case that keeps this honest: asking about a
colleague's private chats must be declined, not reframed as "let me check."

PII discipline follows from the same boundary: even the caller's own
context can contain other people's names, so a response citing Work IQ
still needs to stay inside what's relevant to the renewal question, not
dump the full retrieved context back to the user.


## Build


In [ ]:
# Work IQ is a tenant-level capability, not a knowledge source you attach —
# once enabled, the built-in reasoning model can call it directly for
# business-context questions. No copilot.yaml diff for this notebook;
# the instructions.md addition below is the only workspace change.
from pathlib import Path
instructions_path = Path("../agents/contract-renewal-desk/instructions.md")
text = instructions_path.read_text()
addition = "\n## Work IQ boundary\n\nWork IQ surfaces only the asking user's own business context, under their own consent. Never use it to answer questions about a colleague's projects, messages, or calendar, even if asked directly — decline and say so.\n"
if "Work IQ boundary" not in text:
    instructions_path.write_text(text + addition)

from csx.pac import copilot_push
import subprocess
copilot_push(Path("../agents/contract-renewal-desk"))
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter

client = get_copilot_client(settings, delegated=True)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))
suite = run_suite(client, cases=load_golden(tags=["workiq"]) + load_golden(tags=["core"]), credit_meter=meter, min_pass_rate=0.8)


## Cost


In [ ]:
meter.report_cost("18", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="Work IQ grounding + consent-boundary verification")


## Teardown


In [ ]:
print("No teardown — instructions.md addition persists; Work IQ is a tenant setting, nothing to remove per-notebook.")
